# Xử lý ngôn ngữ tự nhiên - CS221.Q21.KHTN

## Demo chương 19 - Dependency Parsing
## Thành viên:
- Bảo Quý Định Tân - 24520028
- Lê Văn Thức - 24521748
- Lê Phạm Thành Nhân - 24520031

# Demo code tay vui vẻ

In [1]:
class ArcStandardParser:
    """
    Simulates an Arc-Standard transition-based parser.
    """
    def __init__(self, sentence):
        # The parser starts with a stack containing ROOT, and a buffer of words.
        self.stack = ['[ROOT]']
        self.buffer = sentence.split()
        self.relations = []
        
    def shift(self):
        """Remove the word from the front of the input buffer and push it onto the stack."""
        if self.buffer:
            word = self.buffer.pop(0)
            self.stack.append(word)
            print(f"Action: SHIFT -> Stack: {self.stack} | Buffer: {self.buffer}")
            
    def left_arc(self, relation_label):
        """
        Assert a head-dependent relation between the word at the top of the stack 
        and the second word; remove the second word from the stack.
        """
        if len(self.stack) >= 2:
            head = self.stack[-1]
            dependent = self.stack[-2]
            self.relations.append((head, dependent, relation_label))
            self.stack.pop(-2)
            print(f"Action: LEFTARC({relation_label}) -> Added: ({head} -> {dependent})")
            
    def right_arc(self, relation_label):
        """
        Assert a head-dependent relation between the second word on the stack 
        and the word at the top; remove the top word from the stack.
        """
        if len(self.stack) >= 2:
            head = self.stack[-2]
            dependent = self.stack[-1]
            self.relations.append((head, dependent, relation_label))
            self.stack.pop(-1)
            print(f"Action: RIGHTARC({relation_label}) -> Added: ({head} -> {dependent})")

    def show_parse(self):
        print("\nFinal Dependency Relations:")
        for head, dep, label in self.relations:
            print(f"{head} --[{label}]--> {dep}")

# --- Demo Execution ---
print("--- Parsing: 'Book me the flight' ---")
parser = ArcStandardParser("Book me the flight")

# Simulating the Oracle's choices to build the correct parse
parser.shift() # Stack: [ROOT, Book]
parser.shift() # Stack: [ROOT, Book, me]
parser.right_arc("iobj") # book -> me
parser.shift() # Stack: [ROOT, Book, the]
parser.shift() # Stack: [ROOT, Book, the, flight]
parser.left_arc("det") # flight -> the
parser.right_arc("obj") # book -> flight
parser.right_arc("root") # ROOT -> book

parser.show_parse()

--- Parsing: 'Book me the flight' ---
Action: SHIFT -> Stack: ['[ROOT]', 'Book'] | Buffer: ['me', 'the', 'flight']
Action: SHIFT -> Stack: ['[ROOT]', 'Book', 'me'] | Buffer: ['the', 'flight']
Action: RIGHTARC(iobj) -> Added: (Book -> me)
Action: SHIFT -> Stack: ['[ROOT]', 'Book', 'the'] | Buffer: ['flight']
Action: SHIFT -> Stack: ['[ROOT]', 'Book', 'the', 'flight'] | Buffer: []
Action: LEFTARC(det) -> Added: (flight -> the)
Action: RIGHTARC(obj) -> Added: (Book -> flight)
Action: RIGHTARC(root) -> Added: ([ROOT] -> Book)

Final Dependency Relations:
Book --[iobj]--> me
flight --[det]--> the
Book --[obj]--> flight
[ROOT] --[root]--> Book


In [2]:
def evaluate_parser(reference_parse, system_parse):
    """
    Calculates UAS and LAS.
    A relation is a tuple: (head, dependent, label)
    """
    # Unlabeled Attachment Score looks at correctness of assigned head, ignoring relation [cite: 795]
    gold_unlabeled = {(head, dep) for head, dep, label in reference_parse}
    sys_unlabeled = {(head, dep) for head, dep, label in system_parse}
    
    # Labeled Attachment Score requires correct head AND correct relation label [cite: 794]
    gold_labeled = set(reference_parse)
    sys_labeled = set(system_parse)
    
    uas = len(gold_unlabeled.intersection(sys_unlabeled)) / len(gold_unlabeled) if gold_unlabeled else 0
    las = len(gold_labeled.intersection(sys_labeled)) / len(gold_labeled) if gold_labeled else 0
    
    return uas, las

# --- Evaluation Demo ---
# Gold standard from treebank
reference_relations = [
    ("Book", "me", "iobj"),
    ("flight", "the", "det"),
    ("Book", "flight", "obj"),
    ("[ROOT]", "Book", "root")
]

# Simulated system output with a labeling error (xcomp instead of iobj)
system_relations = [
    ("Book", "me", "xcomp"),  # <--- Label error here
    ("flight", "the", "det"),
    ("Book", "flight", "obj"),
    ("[ROOT]", "Book", "root")
]

uas_score, las_score = evaluate_parser(reference_relations, system_relations)

print(f"Unlabeled Attachment Score (UAS): {uas_score * 100:.2f}%")
print(f"Labeled Attachment Score (LAS): {las_score * 100:.2f}%")

Unlabeled Attachment Score (UAS): 100.00%
Labeled Attachment Score (LAS): 75.00%


In [3]:
# --- Code Cell 3: Graph-Based Parsing (Maximum Spanning Tree) ---
# Teacher Note: This simulates the first phase of the Chu-Liu-Edmonds algorithm.
# We create a fully connected graph with arbitrary scores, then greedily 
# select the highest-scoring incoming edge for each dependent.

def greedy_maximum_spanning_tree(vertices, edge_scores):
    """
    vertices: list of words including [ROOT]
    edge_scores: dict of (head, dependent) -> score
    """
    selected_edges = []
    
    # For every word (except ROOT), find the highest scoring incoming edge
    for dependent in vertices:
        if dependent == '[ROOT]':
            continue
            
        best_head = None
        max_score = float('-inf')
        
        for head in vertices:
            if head == dependent:
                continue
                
            # Get the score for this specific head -> dependent directed edge
            score = edge_scores.get((head, dependent), float('-inf'))
            
            if score > max_score:
                max_score = score
                best_head = head
                
        if best_head:
            selected_edges.append((best_head, dependent, max_score))
            
    return selected_edges

# --- Demo Execution ---
words = ['[ROOT]', 'Book', 'that', 'flight']

# Simulated edge scores from a trained neural network/feature classifier
# (Based on the 'Book that flight' example in Chapter 19)
scores = {
    ('[ROOT]', 'Book'): 12, ('[ROOT]', 'that'): 4, ('[ROOT]', 'flight'): 4,
    ('Book', 'that'): 5, ('Book', 'flight'): 7,
    ('that', 'Book'): 6, ('that', 'flight'): 8,
    ('flight', 'Book'): 5, ('flight', 'that'): 7
}

mst_edges = greedy_maximum_spanning_tree(words, scores)

print("Greedy Edge Selection (Maximum Spanning Tree):")
for head, dep, score in mst_edges:
    print(f"Edge: {head} --> {dep} (Score: {score})")
    
# Note to teacher: If this greedy selection resulted in a cycle, 
# the next step of Chu-Liu-Edmonds would be to collapse the cycle into a single node 
# and adjust the weights to eliminate the cycle.

Greedy Edge Selection (Maximum Spanning Tree):
Edge: [ROOT] --> Book (Score: 12)
Edge: flight --> that (Score: 7)
Edge: that --> flight (Score: 8)


In [4]:
def load_mock_treebank():
    """
    Simulates loading a dataset containing sentences and their gold-standard 
    dependency relations.
    """
    dataset = []
    
    # Sentence 1: A purely projective sentence
    # "Book the flight"
    dataset.append({
        "id": "Sent-1-Projective",
        "sentence": "Book the flight",
        "words": ['[ROOT]', 'Book', 'the', 'flight'],
        "gold_relations": [
            ('[ROOT]', 'Book', 'root'),
            ('flight', 'the', 'det'),
            ('Book', 'flight', 'obj')
        ],
        "is_projective": True
    })
    
    # Sentence 2: A non-projective sentence (Simplified from Chapter 19 Eq 19.3)
    # "JetBlue canceled flight morning which was late" 
    # (The arc from 'flight' to 'late' crosses the arc from 'morning' to 'flight')
    dataset.append({
        "id": "Sent-2-NonProjective",
        "sentence": "JetBlue canceled flight morning which was late",
        "words": ['[ROOT]', 'JetBlue', 'canceled', 'flight', 'morning', 'which', 'was', 'late'],
        "gold_relations": [
            ('[ROOT]', 'canceled', 'root'),
            ('canceled', 'JetBlue', 'nsubj'),
            ('canceled', 'flight', 'obj'),
            ('flight', 'morning', 'nmod'),
            ('late', 'which', 'nsubj'),
            ('late', 'was', 'cop'),
            ('flight', 'late', 'acl:relcl') # <--- This causes the non-projective crossing!
        ],
        "is_projective": False
    })
    
    return dataset

treebank_data = load_mock_treebank()
print(f"Loaded {len(treebank_data)} sentences from the mock treebank.")

Loaded 2 sentences from the mock treebank.


In [5]:
def run_comparison(dataset):
    print("==================================================")
    print("  EVALUATION: TRANSITION-BASED vs GRAPH-BASED     ")
    print("==================================================\n")
    
    for data in dataset:
        print(f"--- Analyzing {data['id']} ---")
        print(f"Sentence: '{data['sentence']}'")
        print(f"Structure: {'Projective' if data['is_projective'] else 'Non-Projective'}")
        
        # 1. Simulate Transition-Based Output (Arc-Standard)
        # It gets the projective one perfectly, but makes a forced local error on the non-projective one
        if data['is_projective']:
            arc_standard_output = data['gold_relations'].copy() 
        else:
            # Arc-Standard fails to attach 'late' to 'flight' because 'morning' is in the way.
            # It attaches 'late' to 'canceled' instead due to stack constraints.
            arc_standard_output = data['gold_relations'].copy()
            arc_standard_output.remove(('flight', 'late', 'acl:relcl'))
            arc_standard_output.append(('canceled', 'late', 'advcl')) # Forced Error
            
        # 2. Simulate Graph-Based Output (Maximum Spanning Tree)
        # Because it scores the whole graph, it finds the non-projective tree perfectly.
        graph_based_output = data['gold_relations'].copy()
        
        # Evaluate Arc-Standard
        as_uas, as_las = evaluate_parser(data['gold_relations'], arc_standard_output)
        
        # Evaluate Graph-Based
        gb_uas, gb_las = evaluate_parser(data['gold_relations'], graph_based_output)
        
        # Print Results
        print("\nResults:")
        print(f"  Arc-Standard (Transition) -> UAS: {as_uas*100:.1f}% | LAS: {as_las*100:.1f}%")
        print(f"  Chu-Liu-Edmonds (Graph)   -> UAS: {gb_uas*100:.1f}% | LAS: {gb_las*100:.1f}%")
        if not data['is_projective']:
            print("  *Notice the performance drop in the Transition parser due to the crossing dependency!")
        print("\n")

# Run the experiment
run_comparison(treebank_data)

  EVALUATION: TRANSITION-BASED vs GRAPH-BASED     

--- Analyzing Sent-1-Projective ---
Sentence: 'Book the flight'
Structure: Projective

Results:
  Arc-Standard (Transition) -> UAS: 100.0% | LAS: 100.0%
  Chu-Liu-Edmonds (Graph)   -> UAS: 100.0% | LAS: 100.0%


--- Analyzing Sent-2-NonProjective ---
Sentence: 'JetBlue canceled flight morning which was late'
Structure: Non-Projective

Results:
  Arc-Standard (Transition) -> UAS: 85.7% | LAS: 85.7%
  Chu-Liu-Edmonds (Graph)   -> UAS: 100.0% | LAS: 100.0%
  *Notice the performance drop in the Transition parser due to the crossing dependency!




# Demo sử dụng thư viện

In [6]:
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 53.4 MB/s  0:00:00eta 0:00:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


In [7]:
# 1. Testing with spaCy (Transition-based)
# First, run in terminal: python -m spacy download en_core_web_sm
import spacy

nlp = spacy.load("en_core_web_sm")
doc = nlp("The quick brown fox jumps over the lazy dog.")

print("--- spaCy Dependency Parsing ---")
# token.dep_ is the dependency relation, token.head is the parent word
for token in doc:
    print(f"{token.text:10} | POS: {token.pos_:5} | Dep: {token.dep_:10} | Head: {token.head.text}")


# 2. Testing with Stanza (Graph-based)
# First, run in terminal: pip install stanza
import stanza

# stanza.download('en') # Uncomment to download the model the first time you run this
nlp_stanza = stanza.Pipeline(lang='en', processors='tokenize,mwt,pos,lemma,depparse')
doc_stanza = nlp_stanza("The quick brown fox jumps over the lazy dog.")

print("\n--- Stanza Dependency Parsing ---")
for sentence in doc_stanza.sentences:
    for word in sentence.words:
        # word.head is a 1-based index; head=0 means it's the ROOT of the sentence
        head_word = sentence.words[word.head-1].text if word.head > 0 else "ROOT"
        print(f"{word.text:10} | POS: {word.upos:5} | Dep: {word.deprel:10} | Head: {head_word}")

--- spaCy Dependency Parsing ---
The        | POS: DET   | Dep: det        | Head: fox
quick      | POS: ADJ   | Dep: amod       | Head: fox
brown      | POS: ADJ   | Dep: amod       | Head: fox
fox        | POS: NOUN  | Dep: nsubj      | Head: jumps
jumps      | POS: VERB  | Dep: ROOT       | Head: jumps
over       | POS: ADP   | Dep: prep       | Head: jumps
the        | POS: DET   | Dep: det        | Head: dog
lazy       | POS: ADJ   | Dep: amod       | Head: dog
dog        | POS: NOUN  | Dep: pobj       | Head: over
.          | POS: PUNCT | Dep: punct      | Head: jumps


2026-05-17 11:40:22 INFO: Checking for updates to resources.json in case models have been updated.  Note: this behavior can be turned off with download_method=None or download_method=DownloadMethod.REUSE_RESOURCES


2026-05-17 11:40:22 INFO: Downloaded file to /home/asamai/.cache/stanza/1.12.0/resources/resources.json
2026-05-17 11:40:23 INFO: Loading these models for language: en (English):
| Processor | Package           |
---------------------------------
| tokenize  | combined          |
| mwt       | combined          |
| pos       | combined_charlm   |
| lemma     | combined_nocharlm |
| depparse  | combined_charlm   |

2026-05-17 11:40:23 INFO: Using device: cpu
2026-05-17 11:40:23 INFO: Loading: tokenize
2026-05-17 11:40:23 INFO: Loading: mwt
2026-05-17 11:40:23 INFO: Loading: pos
2026-05-17 11:40:24 INFO: Loading: lemma
2026-05-17 11:40:25 INFO: Loading: depparse
2026-05-17 11:40:25 INFO: Done loading processors!



--- Stanza Dependency Parsing ---
The        | POS: DET   | Dep: det        | Head: fox
quick      | POS: ADJ   | Dep: amod       | Head: fox
brown      | POS: ADJ   | Dep: amod       | Head: fox
fox        | POS: NOUN  | Dep: nsubj      | Head: jumps
jumps      | POS: VERB  | Dep: root       | Head: ROOT
over       | POS: ADP   | Dep: case       | Head: dog
the        | POS: DET   | Dep: det        | Head: dog
lazy       | POS: ADJ   | Dep: amod       | Head: dog
dog        | POS: NOUN  | Dep: obl        | Head: jumps
.          | POS: PUNCT | Dep: punct      | Head: jumps


In [9]:
import spacy
import stanza
from conllu import parse_incr

# 1. Load spaCy (Transition-based)
nlp_spacy = spacy.load("en_core_web_sm", disable=["ner", "lemmatizer", "textcat"])

# 2. Load Stanza (Graph-based)
# We set tokenize_pretokenized=True so we can feed it perfectly split words
nlp_stanza = stanza.Pipeline(
    lang="en", 
    processors="tokenize,pos,lemma,depparse", 
    tokenize_pretokenized=True
)

def compare_parsers(conllu_file_path):
    spacy_uas_correct = 0
    stanza_uas_correct = 0
    total_tokens = 0

    all_gold_tokens = []
    all_sentences_words = []
    
    # 1. Read all gold data and extract raw words
    with open(conllu_file_path, "r", encoding="utf-8") as f:
        for gold_sentence in parse_incr(f):
            # Ignore multi-word tokens (like "don't" -> "do", "n't") for clean evaluation
            tokens = [t for t in gold_sentence if isinstance(t["id"], int)]
            words = [t["form"] for t in tokens]
            
            if not words:
                continue
            
            all_gold_tokens.append(tokens)
            all_sentences_words.append(words)

    # 2. Parse all sentences with Stanza simultaneously (Batching)
    print("Parsing with Stanza (this may take a minute)...")
    # By passing a list of lists, Stanza knows these are pre-tokenized sentences
    stanza_doc = nlp_stanza(all_sentences_words)
    
    # 3. Parse with spaCy and Evaluate both
    print("Evaluating tree structures...")
    from spacy.tokens import Doc
    
    for i, gold_tokens in enumerate(all_gold_tokens):
        words = all_sentences_words[i]
        
        # Parse sentence with spaCy
        doc_spacy = Doc(nlp_spacy.vocab, words=words)
        doc_spacy = nlp_spacy(doc_spacy)
        
        # Get the corresponding Stanza parsed sentence
        stanza_sentence = stanza_doc.sentences[i]
        
        # Compare token by token
        for spacy_token, stanza_token, gold_token in zip(doc_spacy, stanza_sentence.words, gold_tokens):
            gold_head = gold_token["head"]
            
            # spaCy head calculation (spaCy is 0-based, CoNLL is 1-based. 0 means ROOT)
            spacy_head = spacy_token.head.i + 1 if spacy_token.dep_ != "ROOT" else 0
            if spacy_head == gold_head:
                spacy_uas_correct += 1
                
            # Stanza head calculation (already 1-based, 0 means ROOT)
            stanza_head = stanza_token.head
            if stanza_head == gold_head:
                stanza_uas_correct += 1
                
            total_tokens += 1

    # 4. Print Results
    print("-" * 30)
    print(f"Total Tokens Evaluated: {total_tokens}")
    print(f"spaCy UAS (Transition):  {(spacy_uas_correct / total_tokens) * 100:.2f}%")
    print(f"Stanza UAS (Graph):      {(stanza_uas_correct / total_tokens) * 100:.2f}%")

# Run the test
compare_parsers("en_ewt-ud-test.conllu")

2026-05-17 11:42:34 INFO: Checking for updates to resources.json in case models have been updated.  Note: this behavior can be turned off with download_method=None or download_method=DownloadMethod.REUSE_RESOURCES


2026-05-17 11:42:35 INFO: Downloaded file to /home/asamai/.cache/stanza/1.12.0/resources/resources.json
2026-05-17 11:42:35 INFO: Loading these models for language: en (English):
| Processor | Package           |
---------------------------------
| tokenize  | combined          |
| pos       | combined_charlm   |
| lemma     | combined_nocharlm |
| depparse  | combined_charlm   |

2026-05-17 11:42:35 INFO: Using device: cpu
2026-05-17 11:42:35 INFO: Loading: tokenize
2026-05-17 11:42:35 INFO: Loading: pos
2026-05-17 11:42:36 INFO: Loading: lemma
2026-05-17 11:42:37 INFO: Loading: depparse
2026-05-17 11:42:37 INFO: Done loading processors!


Parsing with Stanza (this may take a minute)...
Evaluating tree structures...
------------------------------
Total Tokens Evaluated: 25094
spaCy UAS (Transition):  56.62%
Stanza UAS (Graph):      92.23%


Ta có thể thấy graph-based parser tốt hơn rất nhiều